# G4 Intersect Exploration

Descriptive statistics and visualisations for the `bedtools_intersect` output comparing g4Discovery motifs and rG4detector peaks.

## Research questions

1. Which tool identifies the most G4FS?

    rG4detector identifies a total of 45077 G4 signal peaks, and G4Discovery finds 25654

2. How is the agreement between both tools?
    
    2.1. What percentage of the total G4FS (union) is identified by both tools (intersection)

        - Total motifs captured by RG4D: 25654 / 56094 (45.73%)
        - Total motifs captured by G4D: 45077 / 56094 (80.36%)

    2.2. What percentage of the total G4FS is identified exclusively by one of the tools

        - G4D only motifs: 11017 / 25654 (42.94%)
        - RG4D only motifs: 30279 / 45077 (67.17%)

3. How is the agreement between both tools, when only looking at PC vs LNC transcripts?
    
    

## 1. Setup & Helpers

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from importlib import reload
import sys; sys.path.insert(0, ".")
import notebooks.g4_analysis_lib as lib

SAMPLE = "gencode.v47"
os.chdir("/mnt/cbib/LNClassifier/paper/secondrna")
print("Imports OK")

## 2. Data Loading

In [ ]:
G4DISCOVERY_PATH = f"results/{SAMPLE}/g4discovery/{SAMPLE}.g4Discovery.transcripts.bed.gz"
INTERSECT_PATH   = f"results/{SAMPLE}/{SAMPLE}.g4Discovery.rG4detector.intersect.bed"
PEAKS_PATH       = f"results/{SAMPLE}/{SAMPLE}.rG4detector.peaks.bed"
CODING_ANN_PATH  = f"results/{SAMPLE}/{SAMPLE}.coding_annotation.tsv"
LENGTH_ANN_PATH  = f"results/{SAMPLE}/{SAMPLE}.transcript_lengths.tsv"

g4discovery = lib.load_g4discovery(G4DISCOVERY_PATH)
intersect   = lib.load_intersect(INTERSECT_PATH)
peaks       = lib.load_peaks(PEAKS_PATH)
coding_ann  = lib.load_coding_annotation(CODING_ANN_PATH)
length_ann  = lib.load_length_annotation(LENGTH_ANN_PATH)

print(f"g4discovery: {g4discovery.shape}  intersect: {intersect.shape}  peaks: {peaks.shape}")
print(f"coding_ann:  {coding_ann.shape}   length_ann: {length_ann.shape}")

In [ ]:
union_df = lib.build_union_df(intersect, peaks)
union_df = lib.annotate_union(union_df, coding_ann, length_ann)

assert "real"      in union_df.columns, "missing coding annotation"
assert "tx_len_nt" in union_df.columns, "missing length annotation"
assert set(union_df["type"].unique()) <= {"both", "G4D only", "RG4D only"}
print(f"union_df: {union_df.shape}  types: {union_df['type'].value_counts().to_dict()}")

In [ ]:
subsets = lib.build_subsets(union_df)
for spec, sdf in subsets:
    print(f"{spec.label:40s}  {len(sdf):6,} motifs")
assert len(subsets) == 8

In [ ]:
from IPython.display import display, Markdown
from matplotlib_venn import venn2


def run_subset_analysis(spec, sdf, coding_ann, length_ann):
    """Run all analysis sections for one subset; returns numeric results dict."""
    display(Markdown(f"---\n## Subset: {spec.label}\n`{len(sdf):,}` motifs"))

    # ── 3. Aggregate Statistics ──────────────────────────────────────────────
    display(Markdown("### 3. Aggregate Statistics"))
    agg = lib.compute_aggregate_stats(sdf)
    display(pd.DataFrame([agg]).T.rename(columns={0: "value"}))

    if agg["g4d_total"] > 0 and agg["rg4d_total"] > 0:
        fig, ax = plt.subplots(figsize=(5, 5))
        venn2(
            subsets=(agg["g4d_only"], agg["rg4d_only"], agg["both"]),
            set_labels=("g4Discovery", "rG4detector"),
            ax=ax,
        )
        ax.set_title(f"Tool overlap — {spec.label}")
        plt.tight_layout(); plt.show()

    # ── 4. Coding Breakdown ──────────────────────────────────────────────────
    display(Markdown("### 4. Coding Breakdown"))
    class_bk = lib.compute_coding_breakdown(sdf)
    if not class_bk.empty:
        display(
            class_bk.style.format({
                "Total motifs": "{:,}", "With rG4D overlap": "{:,}", "Overlap (%)": "{:.1f}"
            })
        )
        fig, ax = plt.subplots(figsize=(7, 4))
        x = np.arange(len(class_bk)); w = 0.35
        ax.bar(x - w/2, class_bk["Total motifs"],      w, label="Total",      color=["#1565C0", "#2E7D32"])
        ax.bar(x + w/2, class_bk["With rG4D overlap"], w, label="With rG4D",  color=["#42A5F5", "#66BB6A"])
        ax.set_xticks(x); ax.set_xticklabels(class_bk.index)
        ax.set_ylabel("Motifs"); ax.set_title(f"Coding breakdown — {spec.label}")
        ax.legend(); plt.tight_layout(); plt.show()

    # ── 5. Length & Position ─────────────────────────────────────────────────
    display(Markdown("### 5. Length & Position"))
    pos_start = "rg4d_start" if spec.name == "rg4d_only" else "start"
    pos_end   = "rg4d_end"   if spec.name == "rg4d_only" else "end"

    pos_sdf = sdf.dropna(subset=["tx_len_nt", "real"]).copy()
    pos_sdf = pos_sdf[pos_sdf[pos_start] >= 0]

    if not pos_sdf.empty:
        pos_sdf["relative_pos"] = (
            (pos_sdf[pos_start] + pos_sdf[pos_end]) / 2 / pos_sdf["tx_len_nt"]
        )
        pos_sdf = pos_sdf[pos_sdf["relative_pos"].between(0, 1)]
        pos_sdf["class"] = pos_sdf["real"].map({True: "Protein-coding", False: "Non-coding"})

        fig, ax = plt.subplots(figsize=(8, 4))
        for label, color in [("Protein-coding", "#1565C0"), ("Non-coding", "#2E7D32")]:
            vals = pos_sdf.loc[pos_sdf["class"] == label, "relative_pos"]
            if vals.nunique() > 1:
                sns.kdeplot(vals, ax=ax, label=label, color=color, fill=True, alpha=0.3)
        ax.set_xlabel("Relative position (0 = 5\u2032, 1 = 3\u2032)")
        ax.set_ylabel("Density")
        ax.set_title(f"Positional distribution — {spec.label}")
        ax.legend(); ax.set_xlim(0, 1); plt.tight_layout(); plt.show()
    else:
        print("  (position data unavailable for this subset)")

    # ── 6. Score Analysis ────────────────────────────────────────────────────
    display(Markdown("### 6. Score Analysis"))
    score_col = "rg4d_score" if spec.name == "rg4d_only" else "g4HunterScore"
    score_sdf = sdf[sdf[score_col] > 0].copy()

    if not score_sdf.empty and spec.name != "rg4d_only":
        score_sdf["Overlap"] = score_sdf["overlaps"].map(
            {True: "With rG4D overlap", False: "No rG4D overlap"}
        )
        fig, ax = plt.subplots(figsize=(7, 4))
        sns.violinplot(
            data=score_sdf, x="Overlap", y=score_col,
            hue="Overlap", ax=ax, inner="box", legend=False,
            order=["No rG4D overlap", "With rG4D overlap"],
        )
        ax.set_ylabel("G4Hunter score")
        ax.set_title(f"Score distribution — {spec.label}")
        plt.tight_layout(); plt.show()

    # ── 7. Statistical Tests ─────────────────────────────────────────────────
    display(Markdown("### 7. Statistical Tests"))
    test_res = lib.run_statistical_tests(sdf, coding_ann, length_ann)
    display(pd.DataFrame([test_res]).T.rename(columns={0: "value"}))

    return {
        "spec":           spec,
        "agg_stats":      agg,
        "class_breakdown": class_bk,
        "test_results":   test_res,
    }

In [ ]:
all_results = {}
for spec, sdf in subsets:
    result = run_subset_analysis(spec, sdf, coding_ann, length_ann)
    all_results[spec.name] = result
print(f"\nDone. {len(all_results)} subsets processed.")

In [ ]:
from IPython.display import display, Markdown
display(Markdown("---\n# 8. Cross-Subset Comparison"))

comp_df = lib.compare_results(all_results)
display(
    comp_df[[
        "total_motifs", "g4d_total", "rg4d_total", "both", "g4d_overlap_frac",
        "cramers_v", "p_chi2", "vda", "p_mwu",
        "median_coding_per_kb", "median_lncrna_per_kb",
    ]]
    .style
    .format({
        "total_motifs": "{:,}", "g4d_total": "{:,}", "rg4d_total": "{:,}", "both": "{:,}",
        "g4d_overlap_frac": "{:.3f}",
        "cramers_v": "{:.4f}", "p_chi2": "{:.2e}",
        "vda": "{:.4f}", "p_mwu": "{:.2e}",
        "median_coding_per_kb": "{:.4f}", "median_lncrna_per_kb": "{:.4f}",
    })
    .background_gradient(subset=["cramers_v", "vda"], cmap="YlOrRd")
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comp_df["cramers_v"].plot(
    kind="barh", ax=axes[0], color="#1565C0",
    title="Cramér's V (motif presence/absence association)",
)
axes[0].set_xlabel("Cramér's V  (higher = stronger association)")
axes[0].invert_yaxis()

comp_df["vda"].plot(
    kind="barh", ax=axes[1], color="#2E7D32",
    title="Vargha-Delaney A (motif density shift)",
)
axes[1].axvline(0.5, color="red", linestyle="--", linewidth=1, label="no effect (0.5)")
axes[1].set_xlabel("VDA  (>0.5 = coding has higher density)")
axes[1].legend(fontsize=9)
axes[1].invert_yaxis()

plt.suptitle("Effect sizes by subset — higher is stronger lncRNA/coding discrimination", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
pval_df = comp_df[["p_chi2", "p_mwu"]].copy()
pval_df["-log10(p_chi2)"] = -np.log10(pval_df["p_chi2"].clip(lower=1e-300))
pval_df["-log10(p_mwu)"]  = -np.log10(pval_df["p_mwu"].clip(lower=1e-300))

fig, ax = plt.subplots(figsize=(6, 6))
sns.heatmap(
    pval_df[["-log10(p_chi2)", "-log10(p_mwu)"]],
    annot=True, fmt=".1f", cmap="Blues", ax=ax,
)
ax.set_title("-log\u2081\u2080(p-value): higher = more significant")
plt.tight_layout()
plt.show()